In [1]:
import time
import pandas as pd
from rdkit import Chem
from rdkit import DataStructs
import rdkit.RDLogger as RDLogger
RDLogger.DisableLog('rdApp.*')  # Silence RDKit warnings
from tqdm import tqdm
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Any
import multiprocessing as mp
from functools import partial
from tqdm import tqdm
import numpy as np
from rdchiral.initialization import rdchiralReaction, rdchiralReactants

import sys
import os
sys.path.append('../core')
from ReactChemChecker import *

/apps/external/apps/jupyter/3.1.1/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/apps/external/apps/jupyter/3.1.1/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
2026-02-16 22:15:58.134128: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-16 22:15:58.148658: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-16 22:15:58.153134: E external/local_xla/xla/stream_executor/cuda/

In [5]:
@dataclass
class ModelResult:
    """
    Container for model prediction results
    """
    rank: int = 9999
    top1_match: bool = False
    top1_smiles: Optional[str] = None
    template_found: bool = False
    num_template_matches: int = 0
    template_failed: bool = False

class ReactionSimilarityAnalyzer:
    def __init__(self, fp_pickle_path):

        self.df = pd.read_pickle(fp_pickle_path)
        self.failed_template_check = 0
        self.true_positive = 0
        self.true_negative = 0
        self.false_positive = 0
        self.false_negative = 0
    
    def canonicalize_reaction(self, rxn_smiles):
        """
        Clean and sort reactant/product SMILES for consistent comparisons.
        Splits on '>>', canonicalizes each fragment, and rejoins.
        """
        try:
            reactants, products = rxn_smiles.split('>>')
            # Canonicalize each fragment and sort alphabetically
            canon_reactants = '.'.join(sorted([
                Chem.MolToSmiles(Chem.MolFromSmiles(smi), canonical=True)
                for smi in reactants.split('.')
            ]))
            canon_products = '.'.join(sorted([
                Chem.MolToSmiles(Chem.MolFromSmiles(smi), canonical=True)
                for smi in products.split('.')
            ]))
            return f"{canon_reactants}>>{canon_products}"
        except Exception:
            # If parsing fails, return original
            return rxn_smiles
        
    def _create_default_result(self, ix, row):
        """
        Helper method to create default result for failed instances
        """
        return {
            'Query_Index': ix,
            'RHEA_ID': row.get('RHEA_ID', 'N/A'),
            'Direction': row.get('direction', 'N/A'),
            'Feasible': row['feasible?'],
            'SMILES': row.get('can_rxn_smiles', row.get('rxn_smiles')),
            **self._flatten_model_results('temp', ModelResult())
        }
    
    def _flatten_model_results(self, suffix: str, result: ModelResult) -> Dict[str, Any]:
        """
        Convert ModelResult to flat dictionary
        """
        return {
            f'Rank_{suffix}': result.rank,
            f'Top1_Match_{suffix}': result.top1_match,
            f'Top1_SMILES_{suffix}': result.top1_smiles,
            f'Template_Match_Found_Top1_{suffix}': result.template_found,
            f'Num_Template_Matches_Top1_{suffix}': result.num_template_matches,
            f'Template_Failed_{suffix}': result.template_failed
        }
    
    def _compute_similarities(self, candidate_df: pd.DataFrame, query_rxn_fp, query_prod_fp,
                        query_index: int, query_rhea: str, top_n: int = 50, debug: bool = False) -> List[Tuple[int, float]]:
        """
        Compute similarity scores between query and candidates, with RHEA ID filtering.
        """
        sims = []
        
        for jx, row_j in candidate_df.iterrows():
            
            # Skip if it is the query itself
            if jx == query_index:
                continue
            
            t_rxn_fp = row_j['rxn_fp']
            t_prod_fp = row_j['prod_fp']
            
            if t_rxn_fp is None or t_prod_fp is None:
                continue
                
            # Tanimoto on reaction fingerprint * product fingerprint
            rxn_sim = DataStructs.TanimotoSimilarity(query_rxn_fp, t_rxn_fp)
            prod_sim = DataStructs.TanimotoSimilarity(query_prod_fp, t_prod_fp)
            combined_sim = rxn_sim * prod_sim

            sims.append((jx, combined_sim))
        
        if debug:
            print(f"Computed {len(sims)} similarity scores")
        
        # Sort by similarity in descending order
        sims_sorted = sorted(sims, key=lambda x: x[1], reverse=True)
        
        # Filter out near-neighbor RHEA IDs (±1) - representing same rxn, different directions
        return self._filter_rhea_neighbors(sims_sorted, query_rhea, top_n, debug)
    
    def _filter_rhea_neighbors(self, sims_sorted: List[Tuple[int, float]], 
                              query_rhea: str, top_n: int, debug: bool = False) -> List[Tuple[int, float]]:
        """
        Filter out RHEA IDs that are ±1 from query or each other (as they represent the same reaction in different direction)
        """
        final_valid = []
        
        query_rhea_num = float(query_rhea)
        
        for cidx, sim_score in sims_sorted:
            cand_rhea_num = float(self.df.loc[cidx]['RHEA_ID'])
            
            # Skip if within ±1 of query RHEA
            if abs(cand_rhea_num - query_rhea_num) <= 1:
                continue
            final_valid.append((cidx, sim_score))
            
            if len(final_valid) >= top_n:
                break
        
        if debug:
            print(f"After RHEA-ID filtering: {len(final_valid)} candidates")
        
        return final_valid
    
    def _evaluate_candidates(self, top_n_candidates: List[Tuple[int, float]], 
                           query_condition: bool, debug: bool = False) -> ModelResult:
        """
        Evaluate a list of candidates and return model performance metrics.
        
        Args:
            top_n_candidates: List of (index, similarity_score) tuples
            query_condition: The feasibility condition of the query
            
        Returns:
            ModelResult with computed metrics
        """
        if not top_n_candidates:
            return ModelResult()
        
        # Compute rank (position of first matching feasibility)
        rank = next(
            (i + 1 for i, (cidx, _) in enumerate(top_n_candidates)
             if self.df.loc[cidx]['feasible?'] == query_condition),
            9999
        )
        
        # Top-1 match
        top1_match = self.df.loc[top_n_candidates[0][0]]['feasible?'] == query_condition
            
        if debug:
            print(f"Top 1 match? {top1_match}")
        
        # Extract metadata
        top1_smiles = self.df.loc[top_n_candidates[0][0]].get('can_rxn_smiles')
        
        return ModelResult(
            rank=rank,
            top1_match=top1_match,
            top1_smiles=top1_smiles
        )
        
    def _apply_templates(self, candidates: List[Tuple[int, float]], query_with_cofactor: str, 
                         query_condition: str, debug: bool = False) -> ModelResult:
        """
        Apply reaction templates to candidates and evaluate results.
        """
        template_matches = []
        top1_template_found = False
        
        if debug:
            print(f"\nStarting template application with {len(candidates)} candidates")
            
        query_template = template_extractor(query_with_cofactor)
        
        for i, (cidx, sim_score) in enumerate(candidates):
            if len(template_matches) >= 1:
                if debug:
                    print(f"Reached 1 template matches. Stopping early at candidate {i}")
                break
                
            c_smi = self.df.loc[cidx].get('can_rxn_smiles', self.df.loc[cidx].get('rxn_smiles'))
            c_feas = self.df.loc[cidx].get('feasible?', None)
            
            if debug:
                print(f"\nCandidate {i+1}: Applying template to {c_smi}\nFeasibility marker: {c_feas}")
            
            try:
                temp_app, method = reaction_chemistry_checker(query_template, c_smi)
                if debug:
                    print(f"Template applied?: {temp_app}")
                
                if temp_app:
                    if not top1_template_found:
                        top1_template_found = True
                        if debug:
                            print(f"First template match found at candidate {i+1}")
                    
                    template_matches.append((cidx, sim_score))
                        
                else:
                    if debug:
                        print(f"Template not applied to candidate {i+1}")
                        
            except Exception as e:
                if debug:
                    print(f"Template application failed: {str(e)}")
                continue
        
        if debug:
            print(f"Template application completed. Found {len(template_matches)} matches")
            print(f"Top1 template found: {top1_template_found}")
        
        if not template_matches:
            self.failed_template_check += 1
            
            result = ModelResult()
            result.template_found = top1_template_found
            result.num_template_matches = len(template_matches)
            result.template_failed = True
            return result
        
        # Evaluate template matches
        result = self._evaluate_candidates(template_matches, query_condition, debug=debug)
        result.template_found = top1_template_found
        result.num_template_matches = len(template_matches)
        result.template_failed = False
        
        return result

    def _process_single_query(self, ix: int, row: pd.Series, top_n: int = 50, debug: bool = False) -> Dict[str, Any]:
        """
        Process a single query reaction and return results
        """
        
        query_rxn_fp = row['rxn_fp'] 
        query_prod_fp = row['prod_fp']       
        
        if query_rxn_fp is None or query_prod_fp is None:
            return self._create_default_result(ix, row)
        
        # Extract query metadata
        query_with_cofactor = row['can_rxn_smiles']
        query_condition = row['feasible?']
        query_rhea = row.get('RHEA_ID', 'N/A')
        query_direction = row.get('direction', 'N/A')
        
        if debug:
            print(f"Processing query {ix}, feasibility: {query_condition}")
        
        # 1. Get top similar reactions from knowledgebase
        sims_candidates = self._compute_similarities(
            self.df, query_rxn_fp, query_prod_fp, ix, query_rhea, top_n, debug=debug)
        
        # 2. Template-based model (optional)
        if debug:
            print(f"Applying template-based feasibility model")
        temp_result = ModelResult()
        temp_result = self._apply_templates(sims_candidates, query_with_cofactor, query_condition, debug=debug)

        # Compile results from each model
        result = {
            'Query_Index': ix,
            'RHEA_ID': query_rhea,
            'Direction': query_direction,
            'Feasible': query_condition,
            'SMILES': query_with_cofactor,
            **self._flatten_model_results('temp', temp_result)
        }
        
        if result['Top1_Match_temp']:
            if not result['Template_Failed_temp']:
                if query_condition == "Yes":
                    self.true_positive += 1
                else:
                    self.true_negative += 1
        else:
            if not result['Template_Failed_temp']:
                if query_condition == "No":
                    self.false_positive += 1
                else:
                    self.false_negative += 1

        return result
    
    def run_leave_one_out_cv(self, top_n: int = 50, debug: bool = False, sample_size: Optional[int] = None) -> Tuple[List[Dict], Dict[str, float]]:
        """
        Run leave-one-out cross-validation analysis.
        
        Returns:
            Tuple of (results_list, overall_metrics_dict)
        """
        print("Starting cross-validation analysis...")
        
        if len(self.df) > sample_size:
            query_df = self.df.sample(n=sample_size, random_state=42)  # Use random_state for reproducibility
            print(f"Randomly sampled {sample_size} entries from {len(self.df)} total entries")
        else:
            query_df = self.df
            print(f"Dataset contains {len(self.df)} entries, which is less than requested sample size. Using all entries.")
        
        results = []
        counters = {
            'temp': {'top1': 0}
        }
        
        # Process each query
        for idx, (ix, row) in enumerate(tqdm(query_df.iterrows(), 
                                           total=len(query_df), 
                                           desc="Processing reactions")):
            try:
                result = self._process_single_query(ix, row, top_n, debug)
                results.append(result)
                
                # Update counters
                for model in ['temp']:
                    if result[f'Top1_Match_{model}']:
                        counters[model]['top1'] += 1
                        
            except Exception as e:
                #print(f"Error processing reaction {ix} (iteration {idx}): {str(e)}\nrow:{row['RHEA_ID']}")
                results.append(self._create_default_result(ix, row))
                continue
        
        # Compute overall metrics
        total_without_temp_failures = len(results) - self.failed_template_check
        overall_metrics = {}
        #print(f"Total Fallback count: {self.fallback_count}")
        
        for model in ['temp']:
            overall_metrics[f'Top-1 Accuracy {model.upper()} (%)'] = (
                counters[model]['top1'] / total_without_temp_failures * 100 if total_without_temp_failures else 0
            )
        
        overall_metrics['Total Template App Failure'] = self.failed_template_check
        overall_metrics['Total True Positive (excl. Temp Failures)'] = self.true_positive
        overall_metrics['Total True Negative (excl. Temp Failures)'] = self.true_negative
        overall_metrics['Total False Positive (excl. Temp Failures)'] = self.false_positive
        overall_metrics['Total False Negative (excl. Temp Failures)'] = self.false_negative
        overall_metrics['Total Queries without Temp Application Failure'] = total_without_temp_failures
        overall_metrics['Total Entries in Dataset used'] = len(results)
        
        if debug:
            print("Overall Metrics:", overall_metrics)
        
        return results, overall_metrics
    
    def run_leave_one_out_cv_for_indices(self, indices: List[int], 
                                     top_n: int = 50, debug: bool = False) -> Tuple[List[Dict], Dict[str, float]]:


        #print(f"Starting cross-validation analysis for {len(indices)} specific entries...")
        start_time = time.time()

        # Process only the specified indices, but use full dataset for comparisons
        query_df = self.df.iloc[indices].copy()
        #print(f"Processing {len(query_df)} entries from indices {indices[0]}-{indices[-1]}")

        results = []
        counters = {
            'temp': {'top1': 0}
        }

        for idx, (original_ix, query_row) in enumerate(query_df.iterrows()):
            if debug and idx % 100 == 0:
                print(f"Processing query {idx + 1}/{len(query_df)}")

            try:
                result = self._process_single_query(original_ix, query_row, top_n, debug)
                results.append(result)

                # Update counters
                for model in ['temp']:
                    if result[f'Top1_Match_{model}']:
                        counters[model]['top1'] += 1

            except Exception as e:
                print(f"Error processing reaction {original_ix} (iteration {idx}): {str(e)}\nrow:{query_row['RHEA_ID']}")
                results.append(self._create_default_result(original_ix, query_row))
                continue

        # Calculate metrics for this chunk
        total_without_temp_failures = len(results) - self.failed_template_check
        metrics = {}
        for model in ['temp']:
            metrics[f'Top-1 Accuracy {model.upper()} (%)'] = (
                counters[model]['top1'] / total_without_temp_failures * 100 if total_without_temp_failures else 0
            )

        
        print(f"Total Template App Failure: {self.failed_template_check}")
        metrics['Total Template App Failure'] = self.failed_template_check
        metrics['Total True Positive (excl. Temp Failures)'] = self.true_positive
        metrics['Total True Negative (excl. Temp Failures)'] = self.true_negative
        metrics['Total False Positive (excl. Temp Failures)'] = self.false_positive
        metrics['Total False Negative (excl. Temp Failures)'] = self.false_negative
        metrics['Total Queries without Temp Application Failure'] = total_without_temp_failures
        metrics['Total Entries in Dataset used'] = len(results)

        elapsed_time = time.time() - start_time
        print(f"Completed {len(results)} queries in {elapsed_time:.2f} seconds")

        return results, metrics
        

### Sampled Validation

In [8]:
analyzer = ReactionSimilarityAnalyzer("../data/CV_02152026_directionality_dataset.pkl")
file_name = "Sampled_validation"

try:
    
    results, overall_metrics = analyzer.run_leave_one_out_cv(
        top_n=500,
        debug=False,
        sample_size=1000
    )
    
    # 1) Write the detailed results
    results_df = pd.DataFrame(results)
    results_df.to_csv(f"validation_output/metrics_{file_name}.csv", index=False)
    print("Detailed CV results saved")
    
    # 2) Write the summary metrics
    available_metrics = list(overall_metrics.keys())
    print("Available metrics:", available_metrics)
    
    # 3) Build metrics dataframe
    metrics_data = []
    for metric_name, value in overall_metrics.items():
        metrics_data.append({'Metric': metric_name, 'Score': value})
    
    metrics_df = pd.DataFrame(metrics_data)
    metrics_df.to_csv(f"validation_output/results_{file_name}.csv", index=False)
    print("Overall accuracy metrics saved")
    
except Exception as e:
    print(f"Analysis failed with error: {str(e)}")
    import traceback
    traceback.print_exc()
    raise

Starting cross-validation analysis...
Randomly sampled 1000 entries from 16080 total entries


Processing reactions: 100%|██████████| 1000/1000 [14:01<00:00,  1.19it/s]

Detailed CV results saved
Available metrics: ['Top-1 Accuracy TEMP (%)', 'Total Template App Failure', 'Total True Positive (excl. Temp Failures)', 'Total True Negative (excl. Temp Failures)', 'Total False Positive (excl. Temp Failures)', 'Total False Negative (excl. Temp Failures)', 'Total Queries without Temp Application Failure', 'Total Entries in Dataset used']
Overall accuracy metrics saved


### Parallelized Validation

In [6]:
def process_cv_chunk(chunk_indices, analyzer, top_n, debug):
    
    process_id = mp.current_process().pid
    print(f"[Process {process_id}] Processing {len(chunk_indices)} entries (indices {chunk_indices[0]}-{chunk_indices[-1]})...")
    
    try:
        results, metrics = analyzer.run_leave_one_out_cv_for_indices(
            indices=chunk_indices,
            top_n=top_n,
            debug=debug
        )
        
        print(f"[Process {process_id}] Finished: {len(results)} results")
        return results, metrics
        
    except Exception as e:
        print(f"[Process {process_id}] Error: {e}")
        return [], {}
    
def parallelize_cv_analysis(analyzer, n_processes=None, top_n=15, debug=False):
    """
    Parallelize CV analysis across multiple processes.
    
    Args:
        analyzer: ReactionSimilarityAnalyzer instance
        n_processes: Number of processes to use. If None, auto-detects.
        top_n: Number of top candidates to consider
        debug: Enable debug output
    
    Returns:
        Tuple of (all_results, combined_metrics)
    """
    
    total_entries = len(analyzer.df)
    print(f"Total dataset size: {total_entries} entries")
    
    # Get number of processes
    if n_processes is None:
        try:
            n_processes = len(os.sched_getaffinity(0))
        except:
            n_processes = 1
        
    # Split indices into chunks
    all_indices = list(range(total_entries))
    chunk_size = total_entries // n_processes
    
    index_chunks = []
    for i in range(n_processes):
        start_idx = i * chunk_size
        if i == n_processes - 1:  # Last process gets remaining entries
            end_idx = total_entries
        else:
            end_idx = (i + 1) * chunk_size
        
        chunk_indices = all_indices[start_idx:end_idx]
        index_chunks.append(chunk_indices)
    
    chunk_sizes = [len(chunk) for chunk in index_chunks]
    print(f"Splitting {total_entries} entries across {n_processes} processes: {chunk_sizes}")
    
    # double check - check if condition returns true
    total_covered = sum(chunk_sizes)
    assert total_covered == total_entries, f"Mismatch: {total_covered} != {total_entries}"
    
    # partial function
    process_func = partial(
        process_cv_chunk,
        analyzer=analyzer,
        top_n=top_n,
        debug=debug
    )
            
    print(f"Processing CV analysis using {n_processes} processes")
    with mp.Pool(processes=n_processes) as pool:
        results = list(tqdm(
            pool.imap(process_func, index_chunks),
            total=len(index_chunks),
            desc="Processing CV chunks"
        ))
    
    # combine results
    all_results = []
    all_metrics = []
    
    for results_chunk, metrics_chunk in results:
        if results_chunk:  
            all_results.extend(results_chunk)
            all_metrics.append(metrics_chunk)
            
    print(f"Combined results: {len(all_results)} total entries processed")
    combined_metrics = combine_cv_metrics(all_results, all_metrics)
    
    return all_results, combined_metrics

def combine_cv_metrics(all_results, metrics_list):
    
    if not all_results:
        return {}
    
    total_fallbacks = 0
    total_true_positive = 0
    total_true_negative = 0
    total_false_positive = 0
    total_false_negative = 0
    total_without_failures = 0
    total_queries = 0
    
    if metrics_list:
        for metrics in metrics_list:
            total_fallbacks += metrics['Total Template App Failure']
            total_true_positive += metrics['Total True Positive (excl. Temp Failures)']
            total_true_negative += metrics['Total True Negative (excl. Temp Failures)']
            total_false_positive += metrics['Total False Positive (excl. Temp Failures)']
            total_false_negative += metrics['Total False Negative (excl. Temp Failures)']
            total_without_failures += metrics['Total Queries without Temp Application Failure']
            total_queries += metrics['Total Entries in Dataset used']
    
    counters = {
        'temp': {'top1': 0}
    }
    
    for result in all_results:
        for model in ['temp']:
            if result[f'Top1_Match_{model}']:
                counters[model]['top1'] += 1
    
    # Build combined metrics
    combined_metrics = {}
    
    for model in ['temp']:
        combined_metrics[f'Top-1 Accuracy {model.upper()} (%)'] = (
            counters[model]['top1'] / total_without_failures * 100 if total_without_failures > 0 else 0
        )
    
    combined_metrics['Total Template App Failure'] = total_fallbacks
    combined_metrics['Total True Positive (excl. Temp Failures)'] = total_true_positive
    combined_metrics['Total True Negative (excl. Temp Failures)'] = total_true_negative
    combined_metrics['Total False Positive (excl. Temp Failures)'] = total_false_positive
    combined_metrics['Total False Negative (excl. Temp Failures)'] = total_false_negative
    combined_metrics['Total Queries without Temp Application Failure'] = total_without_failures
    
    combined_metrics['Precision'] = (total_true_positive / (total_true_positive + total_false_positive)) * 100 if (total_true_positive + total_false_positive) > 0 else 0
    combined_metrics['Recall'] = (total_true_positive / (total_true_positive + total_false_negative)) * 100 if (total_true_positive + total_false_negative) > 0 else 0
    
    combined_metrics['Total Entries in Dataset used'] = total_queries
    
    return combined_metrics

In [7]:
analyzer = ReactionSimilarityAnalyzer("../data/CV_02152026_directionality_dataset.pkl")
file_name = "CV_02152026_directionality_dataset"    
try:
    # Run parallelized CV analysis
    results, overall_metrics = parallelize_cv_analysis(analyzer, n_processes=None, 
                                  top_n=500, debug=False)

    # 1) Write the detailed results
    results_df = pd.DataFrame(results)
    results_df.to_csv(f"validation_output/results_{file_name}.csv", index=False)
    print("Detailed CV results saved")

    # 2) Write the summary metrics
    available_metrics = list(overall_metrics.keys())
    print("Available metrics:", available_metrics)

    # 3) Build metrics dataframe 
    metrics_data = []
    for metric_name, value in overall_metrics.items():
        metrics_data.append({'Metric': metric_name, 'Score': value})

    metrics_df = pd.DataFrame(metrics_data)
    metrics_df.to_csv(f"validation_output/metrics_{file_name}.csv", index=False)
    print("Overall accuracy metrics saved ")

except Exception as e:
    print(f"Analysis failed with error: {str(e)}")
    import traceback
    traceback.print_exc()
    raise

Total dataset size: 16080 entries
Splitting 16080 entries across 30 processes: [536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536, 536]
Processing CV analysis using 30 processes


Processing CV chunks:   0%|          | 0/30 [00:00<?, ?it/s]

[Process 4013044] Processing 536 entries (indices 0-535)...
[Process 4013045] Processing 536 entries (indices 536-1071)...
[Process 4013046] Processing 536 entries (indices 1072-1607)...
[Process 4013047] Processing 536 entries (indices 1608-2143)...
[Process 4013048] Processing 536 entries (indices 2144-2679)...
[Process 4013049] Processing 536 entries (indices 2680-3215)...
[Process 4013050] Processing 536 entries (indices 3216-3751)...
[Process 4013051] Processing 536 entries (indices 3752-4287)...
[Process 4013052] Processing 536 entries (indices 4288-4823)...
[Process 4013053] Processing 536 entries (indices 4824-5359)...
[Process 4013054] Processing 536 entries (indices 5360-5895)...
[Process 4013055] Processing 536 entries (indices 5896-6431)...
[Process 4013056] Processing 536 entries (indices 6432-6967)...
[Process 4013057] Processing 536 entries (indices 6968-7503)...
[Process 4013058] Processing 536 entries (indices 7504-8039)...
[Process 4013059] Processing 536 entries (ind

Processing CV chunks:   3%|▎         | 1/30 [07:58<3:51:07, 478.18s/it]

Total Template App Failure: 2
Completed 536 queries in 466.10 seconds
[Process 4013073] Finished: 536 results
Total Template App Failure: 5
Completed 536 queries in 471.94 seconds
[Process 4013061] Finished: 536 results
Total Template App Failure: 4
Completed 536 queries in 477.39 seconds
[Process 4013054] Finished: 536 results
Total Template App Failure: 2
Completed 536 queries in 477.09 seconds
[Process 4013055] Finished: 536 results
Total Template App Failure: 0
Completed 536 queries in 479.25 seconds
[Process 4013053] Finished: 536 results
Total Template App Failure: 3
Completed 536 queries in 474.98 seconds
[Process 4013070] Finished: 536 results
Total Template App Failure: 5
Completed 536 queries in 487.07 seconds
[Process 4013046] Finished: 536 results
Total Template App Failure: 8
Completed 536 queries in 487.26 seconds
[Process 4013048] Finished: 536 results
Total Template App Failure: 2
Completed 536 queries in 481.09 seconds
[Process 4013062] Finished: 536 results
Total Temp

...

Total Template App Failure: 7
Completed 536 queries in 530.42 seconds
[Process 4013065] Finished: 536 results


Processing CV chunks:  73%|███████▎  | 22/30 [09:00<01:26, 10.86s/it]

Total Template App Failure: 7
Completed 536 queries in 538.47 seconds
[Process 4013068] Finished: 536 results


Processing CV chunks: 100%|██████████| 30/30 [09:09<00:00, 18.32s/it]


Combined results: 16080 total entries processed
Detailed CV results saved
Available metrics: ['Top-1 Accuracy TEMP (%)', 'Total Template App Failure', 'Total True Positive (excl. Temp Failures)', 'Total True Negative (excl. Temp Failures)', 'Total False Positive (excl. Temp Failures)', 'Total False Negative (excl. Temp Failures)', 'Total Queries without Temp Application Failure', 'Precision', 'Recall', 'Total Entries in Dataset used']
Overall accuracy metrics saved 
